# Notebook 5 | Product Rule of Probability

In [1]:
from foundations_of_probability_and_statistics.cars.car_distribution import create_car_distribution
from foundations_of_probability_and_statistics.cars.car_distribution import create_joint_car_distribution

import pandas as pd

# show all rows of data frames and series per default
pd.set_option("display.max_rows", None)

## The product rule: from conditionals back to the joint

Given a conditional and the marginal it conditions on, how do we rebuild the joint? Rearranging the definition of conditional probability from notebook 3 gives both symmetric directions; for generic $X$ and $Y$ with values $x$ and $y$,

$$p(X = x, Y = y) = p(Y = y) \, p(X = x \mid Y = y) = p(X = x) \, p(Y = y \mid X = x).$$

Reach for it whenever a joint entry must be assembled from available pieces rather than read off a table. Both directions describe the same entry, so their agreement is a built-in consistency check -- and the notebook 1 factorization is simply this rule applied twice, once per added variable.

## Derivation

1. Start from $p(x \mid y) = p(x, y) / p(y)$ (notebook 3) and multiply both sides by $p(y)$.
2. This gives $p(x, y) = p(y) \, p(x \mid y)$, one direction of the product rule.
3. Swap the roles of $X$ and $Y$ for the symmetric direction $p(x, y) = p(x) \, p(y \mid x)$.
4. For the cars, group $X = (B, H)$ and $Y = C$: both directions reproduce the notebook 1 entry $p(b, h, c) = p(b) \, p(h \mid b) \, p(c \mid b)$, with $p(c \mid b, h) = p(c \mid b)$ by the conditional independence of notebook 4.

The characteristic mistake is multiplying a marginal with a conditional from a mismatched grouping, e.g. $p(b) \, p(c)$ where $p(c \mid b)$ is required.

## Worked example (by hand)

Both symmetric directions on the notebook 1 number $p(\text{Porsche}, 600, \text{black}) = 0.006$, grouping $X = (B, H)$ and $Y = C$:

$$p(\text{Porsche}, 600, \text{black}) = p(\text{Porsche}, 600) \, p(\text{black} \mid \text{Porsche}, 600) = 0.015 \cdot 0.4 = 0.006,$$
$$p(\text{Porsche}, 600, \text{black}) = p(\text{black}) \, p(\text{Porsche}, 600 \mid \text{black}) = 0.31 \cdot (0.006 / 0.31) = 0.006,$$

where $p(\text{black} \mid \text{Porsche}, 600) = p(\text{black} \mid \text{Porsche})$ drops the horsepower by conditional independence. The intermediate $p(\text{Porsche}, 600) = 0.3 \cdot 0.05 = 0.015$ is itself a product-rule step, previewing the chain rule of notebook 6.

In [2]:
import math

joint = create_joint_car_distribution()
car_distribution = create_car_distribution()

# first direction: p(b, h, c) = p(b, h) p(c | b, h) with p(black | Porsche, 600) = p(black | Porsche)
p_porsche_600 = joint.loc[("Porsche", 600, slice(None))].sum()
assert math.isclose(p_porsche_600, 0.3 * 0.05)
assert math.isclose(p_porsche_600, 0.015)
assert math.isclose(joint.loc[("Porsche", 600, "black")], p_porsche_600 * 0.4)
assert math.isclose(joint.loc[("Porsche", 600, "black")], 0.006)

# symmetric direction: p(b, h, c) = p(c) p(b, h | c)
p_black = joint.xs("black", level="color").sum()
assert math.isclose(p_black, 0.31)
assert math.isclose(joint.loc[("Porsche", 600, "black")], p_black * (0.006 / p_black))
joint.loc[("Porsche", 600, "black")]

np.float64(0.006)

## Generalization

Applying the product rule column by column rebuilds the whole joint table from the brand marginal and the per-brand conditionals, which is exactly how `create_joint_car_distribution` is implemented. Notebook 6 extends the same peeling gesture to arbitrary orderings -- the chain rule.

In [3]:
rebuilt = {
    (car_distribution.brand_names[b], int(h), car_distribution.color_names[c]): car_distribution.brand_rv.pmf(b)
    * car_distribution.horsepower_rvs[b].pmf(h)
    * car_distribution.color_rvs[b].pmf(c)
    for b in car_distribution.horsepower_rvs
    for h in car_distribution.horsepower_rvs[b].xk
    for c in car_distribution.color_rvs[b].xk
}

# every rebuilt state matches the joint table
assert all(math.isclose(joint.loc[key], value) for key, value in rebuilt.items())
assert math.isclose(sum(rebuilt.values()), 1.0)
len(rebuilt)

38

## References

- Blitzstein, J. K., Hwang, J. (2019): "Introduction to Probability", 2nd ed., Chapman & Hall/CRC, chapter "Conditional Probability", https://www.routledge.com/Introduction-to-Probability-Second-Edition/Blitzstein-Hwang/p/book/9781138369917 (free PDF: https://probabilitybook.net/).
- Wasserman, L. (2004): "All of Statistics: A Concise Course in Statistical Inference", Springer Texts in Statistics, chapter "Probability", https://doi.org/10.1007/978-0-387-21736-9.
- scipy.stats.rv_discrete, https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_discrete.html.